# 01 — Set up the LearnSphere dataset

Creates the BigQuery dataset and the **five** source tables the sync pipelines
read from:

| Table | Grain | Purpose |
| --- | --- | --- |
| `learners` | one row per learner | Profile + plan tier |
| `instructors` | one row per instructor | Profile + global rating |
| `courses` | one row per course | Course catalog, FK → `instructors` |
| `enrollments` | one row per (learner, course) | Lifecycle + progress |
| `course_reviews` | one row per review | Ratings + comments |

The downstream sync joins these tables in **BigQuery** and ships the denormalized
result to Cosmos DB. Keep schema changes here in sync with the pipeline SQL under
`py/apps/bq-cosmos-sync/src/bq_cosmos_sync/pipelines/`.


In [ ]:
# Load .env from the repo root so we use the same settings as the sync job.
from pathlib import Path
import os

from dotenv import load_dotenv

repo_root = Path.cwd()
while not (repo_root / ".env.example").exists():
    if repo_root == repo_root.parent:
        raise RuntimeError("could not find repo root (.env.example missing)")
    repo_root = repo_root.parent

load_dotenv(repo_root / ".env")

BQ_PROJECT = os.environ["BQ_PROJECT_ID"]
BQ_DATASET = os.environ.get("BQ_DATASET", "learnsphere")
BQ_LOCATION = os.environ.get("BQ_LOCATION", "US")
print(f"project={BQ_PROJECT}  dataset={BQ_DATASET}  location={BQ_LOCATION}")


In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=BQ_PROJECT, location=BQ_LOCATION)

dataset_ref = bigquery.DatasetReference(BQ_PROJECT, BQ_DATASET)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = BQ_LOCATION
dataset = bq.create_dataset(dataset, exists_ok=True)
print(f"dataset ready: {dataset.full_dataset_id}")


## Table schemas


In [ ]:
SCHEMAS = {
    "learners": [
        bigquery.SchemaField("learner_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("display_name", "STRING"),
        bigquery.SchemaField("country", "STRING"),
        bigquery.SchemaField("signup_date", "DATE"),
        bigquery.SchemaField("plan_tier", "STRING"),
        bigquery.SchemaField("updated_at", "TIMESTAMP", mode="REQUIRED"),
    ],
    "instructors": [
        bigquery.SchemaField("instructor_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("display_name", "STRING"),
        bigquery.SchemaField("bio", "STRING"),
        bigquery.SchemaField("country", "STRING"),
        bigquery.SchemaField("rating", "FLOAT64"),
        bigquery.SchemaField("updated_at", "TIMESTAMP", mode="REQUIRED"),
    ],
    "courses": [
        bigquery.SchemaField("course_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("title", "STRING"),
        bigquery.SchemaField("category", "STRING"),
        bigquery.SchemaField("level", "STRING"),
        bigquery.SchemaField("duration_minutes", "INT64"),
        bigquery.SchemaField("price_usd", "FLOAT64"),
        bigquery.SchemaField("instructor_id", "STRING"),
        bigquery.SchemaField("updated_at", "TIMESTAMP", mode="REQUIRED"),
    ],
    "enrollments": [
        bigquery.SchemaField("enrollment_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("learner_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("course_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("enrolled_at", "TIMESTAMP"),
        bigquery.SchemaField("progress_percent", "FLOAT64"),
        bigquery.SchemaField("status", "STRING"),
        bigquery.SchemaField("last_activity_at", "TIMESTAMP"),
    ],
    "course_reviews": [
        bigquery.SchemaField("review_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("course_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("learner_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("rating", "INT64"),
        bigquery.SchemaField("comment", "STRING"),
        bigquery.SchemaField("created_at", "TIMESTAMP"),
    ],
}

for name, schema in SCHEMAS.items():
    table = bigquery.Table(dataset.table(name), schema=schema)
    bq.create_table(table, exists_ok=True)
    print(f"  ok  {name}")


Next: `02_seed_data.ipynb` populates these tables with deterministic fake data.
